# 07a — ChemBench4K Knowledge Probe (Base + LoRA)

**Phase 7a** of the FYP. Tests whether LoRA fine-tuning preserved general chemistry knowledge.

**Goal:** score each model on ChemBench4K (chemistry MCQ) in two states:
1. **Base model** (no adapter) → reference knowledge level
2. **LoRA adapter attached** (best Phase 5 config per (model, dataset))

**12 evaluations:** 3 bases + 9 LoRA adapters (3 models × 3 datasets).

**Hypothesis:** if LoRA AUC scores in Phase 5 came from real chemistry learning, ChemBench4K should hold or improve. If it crashes, the adapter only memorised the task.

**Output:** `chembench_results.csv` (12 rows) saved to Drive.

Phase 7b will add the 9 QLoRA evaluations. Self-contained for Colab VM resets.

## 1. Environment + Drive mount

In [ ]:
%pip install -q --upgrade torchao
%pip install -q --upgrade transformers peft accelerate bitsandbytes datasets rdkit pandas scikit-learn

In [ ]:
import os, sys, time, json, gc, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/chemistry-peft-fyp')
RESULTS_DIR  = PROJECT_ROOT / 'results'
ADAPTERS_DIR = PROJECT_ROOT / 'adapters'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CHEMBENCH_CSV = RESULTS_DIR / 'chembench_results.csv'
print('Results will go to:', CHEMBENCH_CSV)

## 2. Load ChemBench4K from Hugging Face

Uses the public `AI4Chem/ChemBench4K` benchmark (4-choice MCQ across chemistry sub-areas).

In [ ]:
from datasets import load_dataset

CHEMBENCH_ID = 'AI4Chem/ChemBench4K'
raw = load_dataset(CHEMBENCH_ID, split='test')
print('ChemBench4K size:', len(raw))
print('Fields:', raw.column_names)
print()
print('Sample:')
print(raw[0])

In [ ]:
def normalise_example(ex):
    q = ex.get('question') or ex.get('Question') or ex.get('prompt') or ''
    options = ex.get('options') or ex.get('choices') or [ex.get('A',''), ex.get('B',''), ex.get('C',''), ex.get('D','')]
    if isinstance(options, dict):
        options = [options.get(k,'') for k in ['A','B','C','D']]
    options = [str(o) for o in options[:4]]
    answer = ex.get('answer') or ex.get('Answer') or ex.get('label') or ''
    if isinstance(answer, int):
        answer = 'ABCD'[answer]
    answer = str(answer).strip().upper()[:1] if answer else ''
    return {'question': q, 'options': options, 'answer': answer}

rows = [normalise_example(ex) for ex in raw]
rows = [r for r in rows if r['answer'] in {'A','B','C','D'} and len(r['options']) == 4]
print(f'Usable MCQ rows: {len(rows)}')
print('Sample:', rows[0])

## 3. MCQ evaluation harness

Builds a prompt of the form:

```
Question: ...
A) ...
B) ...
C) ...
D) ...
Answer:
```

Then reads logits at the answer position for tokens A/B/C/D and picks the argmax. This is identical in spirit to the Phase 4 yes/no logit scoring — no free-form generation, no regex parsing surprises.

In [ ]:
MCQ_TEMPLATE = (
    'You are a chemistry expert. Choose the best answer.\n\n'
    'Question: {q}\n'
    'A) {a}\nB) {b}\nC) {c}\nD) {d}\n'
    'Answer:'
)

def build_prompt(row):
    a, b, c, d = row['options']
    return MCQ_TEMPLATE.format(q=row['question'], a=a, b=b, c=c, d=d)

def letter_token_ids(tokenizer):
    ids = {}
    for letter in ['A','B','C','D']:
        candidates = [letter, ' ' + letter]
        chosen = None
        for cand in candidates:
            toks = tokenizer.encode(cand, add_special_tokens=False)
            if len(toks) == 1:
                chosen = toks[0]; break
        if chosen is None:
            chosen = tokenizer.encode(letter, add_special_tokens=False)[0]
        ids[letter] = chosen
    return ids

@torch.no_grad()
def score_mcq(model, tokenizer, rows, device, max_len=512):
    model.eval()
    letter_ids = letter_token_ids(tokenizer)
    correct = 0
    total = 0
    t0 = time.time()
    for i, r in enumerate(rows):
        prompt = build_prompt(r)
        enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=max_len).to(device)
        logits = model(**enc).logits[0, -1]
        scored = {L: float(logits[letter_ids[L]]) for L in 'ABCD'}
        pred = max(scored, key=scored.get)
        if pred == r['answer']:
            correct += 1
        total += 1
        if (i+1) % 200 == 0:
            rate = (i+1) / (time.time() - t0)
            print(f'  {i+1}/{len(rows)} | acc={correct/total:.3f} | {rate:.1f} q/s')
    return correct / total, total

## 4. Model + adapter registry

Phase 5 saved adapters under `/MyDrive/chemistry-peft-fyp/adapters/lora/<short>/<dataset>/`.
If a directory is missing, that adapter is skipped and reported.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_CONFIGS = {
    'microsoft/Phi-4-mini-instruct': {'short': 'phi4'},
    'HuggingFaceTB/SmolLM3-3B':      {'short': 'smollm3'},
    'gpt2':                          {'short': 'gpt2'},
}
DATASETS = ['BBBP', 'BACE', 'ESOL']

def adapter_dir(model_name, dataset):
    short = MODEL_CONFIGS[model_name]['short']
    return ADAPTERS_DIR / 'lora' / short / dataset

def load_base(model_name):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, trust_remote_code=True
    ).to('cuda' if torch.cuda.is_available() else 'cpu')
    return base, tok

def cleanup(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Results helper — append-on-write

Every evaluation appends one row to `chembench_results.csv` immediately, so a Colab disconnect mid-loop only loses the in-flight run.

In [ ]:
RESULT_COLS = ['model','variant','dataset','accuracy','n_questions','time_s']

def init_results_csv():
    if not CHEMBENCH_CSV.exists():
        pd.DataFrame(columns=RESULT_COLS).to_csv(CHEMBENCH_CSV, index=False)

def append_result(row):
    df = pd.DataFrame([row])
    df.to_csv(CHEMBENCH_CSV, mode='a', header=False, index=False)
    print(f'  saved -> {row}')

def already_done(model, variant, dataset):
    if not CHEMBENCH_CSV.exists():
        return False
    df = pd.read_csv(CHEMBENCH_CSV)
    return ((df['model']==model) & (df['variant']==variant) & (df['dataset']==dataset)).any()

init_results_csv()
print('Current rows:', len(pd.read_csv(CHEMBENCH_CSV)))

## 6. Score base models (3 runs)

`variant=base`, `dataset=N/A`.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

for model_name in MODEL_CONFIGS:
    if already_done(model_name, 'base', 'N/A'):
        print(f'[skip] base {model_name} already scored')
        continue
    print(f'\n=== Base: {model_name} ===')
    base, tok = load_base(model_name)
    t0 = time.time()
    acc, n = score_mcq(base, tok, rows, device)
    elapsed = time.time() - t0
    append_result({
        'model': model_name,
        'variant': 'base',
        'dataset': 'N/A',
        'accuracy': round(acc, 4),
        'n_questions': n,
        'time_s': round(elapsed, 1),
    })
    cleanup(base, tok)

## 7. Score LoRA adapters (9 runs)

For each (model, dataset) we attach the Phase 5 LoRA adapter and re-score ChemBench4K.

In [ ]:
for model_name in MODEL_CONFIGS:
    for ds in DATASETS:
        if already_done(model_name, 'lora', ds):
            print(f'[skip] LoRA {model_name}/{ds} already scored')
            continue
        adir = adapter_dir(model_name, ds)
        if not adir.exists():
            print(f'[miss] adapter not found: {adir}')
            append_result({
                'model': model_name, 'variant': 'lora', 'dataset': ds,
                'accuracy': float('nan'), 'n_questions': 0, 'time_s': 0.0,
            })
            continue
        print(f'\n=== LoRA: {model_name} / {ds} ===')
        base, tok = load_base(model_name)
        model = PeftModel.from_pretrained(base, str(adir))
        t0 = time.time()
        acc, n = score_mcq(model, tok, rows, device)
        elapsed = time.time() - t0
        append_result({
            'model': model_name,
            'variant': 'lora',
            'dataset': ds,
            'accuracy': round(acc, 4),
            'n_questions': n,
            'time_s': round(elapsed, 1),
        })
        cleanup(model, base, tok)

## 8. Final scoreboard view

In [ ]:
final = pd.read_csv(CHEMBENCH_CSV)
print(f'ChemBench4K rows so far: {len(final)}\n')
print(final.to_string(index=False))

## 9. Knowledge-erosion check (LoRA vs base, per model)

In [ ]:
if {'accuracy','variant','model'}.issubset(final.columns):
    base_acc = final[final['variant']=='base'].set_index('model')['accuracy']
    lora_rows = final[final['variant']=='lora'].copy()
    lora_rows['base_acc']  = lora_rows['model'].map(base_acc)
    lora_rows['delta_acc'] = lora_rows['accuracy'] - lora_rows['base_acc']
    cols = ['model','dataset','base_acc','accuracy','delta_acc']
    print('Knowledge erosion vs base:')
    print(lora_rows[cols].to_string(index=False))
    print()
    print(f'Average delta (LoRA - base): {lora_rows["delta_acc"].mean():.4f}')